<div style="background: linear-gradient(135deg, #080c14 0%, #111d31 50%, #1a0a2e 100%); padding: 40px; border-radius: 16px; text-align: center; margin-bottom: 20px;">
  <h1 style="color: #4f8ef7; font-size: 2.2em; margin: 0; font-family: 'Segoe UI', sans-serif;">🎙️ LiveAvatar Streaming Pipeline</h1>
  <h2 style="color: #8b5cf6; font-size: 1.2em; font-weight: 300; margin: 10px 0 0 0;">Moshi · Bridge · Ditto — Real-Time Talking Head  <span style="color:#22c55e;font-size:.8em;">v2 (Production)</span></h2>
  <p style="color: #64748b; margin: 14px 0 0 0; font-size: 0.9em;">
    Browser Mic → Moshi Streaming → Bridge (KV-cache) → Ditto Online Mode → Live Video + Audio
  </p>
</div>

---

### 📋 What's new in v2 (production fixes)

| Fix | Problem Solved |
|-----|----------------|
| `await put()` + timeout on token_queue | Silent token drops with `put_nowait` |
| Pre-patch Ditto writer **before** `setup()` | Thread race: old+new writer both running |
| `put_nowait` + drop in Ditto writer | Deadlock when browser disconnects |
| `run_coroutine_threadsafe` for frame_queue | asyncio.Queue called from non-async thread |
| Adaptive bridge flush (100ms timeout) | 320ms+ stall during pauses |
| Seq numbers on audio & video packets | No audio-video synchronisation |
| Shared `error_event` across tasks | Tasks hang after another task crashes |
| Moshi busy → immediate WS rejection | Second session silently queued forever |
| Dedicated CUDA stream for Bridge | Implicit CUDA sync stalls |
| TurboJPEG / cv2 JPEG encoding | PIL JPEG ~3-5ms → <1ms per frame |

> **Requirements**: RunPod GPU ≥ A100 40 GB.  
> **Audio**: server sends raw float32 PCM → browser AudioWorklet plays directly.  
> **A/V Sync**: 4-byte seq headers on both audio and video packets align lip-sync in the browser.

In [ ]:
!git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/moshi-ditto-streaming-v3.git

In [ ]:
%cd /workspace/moshi-ditto-streaming-v3

---
## 🔧 Cell 1 — Install Dependencies

Run **once** per RunPod session.

In [ ]:
import os, subprocess, sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

install_script = os.path.join(PROJECT_ROOT, "install.sh")
if not os.path.isfile(install_script):
    raise FileNotFoundError(f"install.sh not found at: {install_script}")

print("🚀 Starting installation...\n")
result = subprocess.run(["bash", install_script], cwd=PROJECT_ROOT, text=True)
if result.returncode != 0:
    print("❌ Installation failed — check output above.")
else:
    print("\n✅ Installation complete!")

In [ ]:
# System packages + cuDNN (needed for TRT and ONNX CUDA provider)
!apt update -qq
!apt-get install -y ffmpeg libcudnn8 libcudnn8-dev 2>&1 | tail -5

In [ ]:
# Optional: install libjpeg-turbo Python binding for fastest JPEG encoding
# The server auto-detects it; falls back to cv2 or PIL if not installed.
# On A100 RunPod this gives ~0.5ms/frame vs ~3ms with PIL.
!pip install -q PyTurboJPEG 2>&1 | tail -3
try:
    from turbojpeg import TurboJPEG
    print("✅ TurboJPEG available — fastest JPEG encoding enabled")
except ImportError:
    print("ℹ️  TurboJPEG not available — will use cv2 or PIL (still functional)")

In [ ]:
# TensorRT (for Ditto)
!pip install -q filetype cython huggingface_hub
!pip install -q cuda-python==12.4.0
!python -c "from cuda import cuda, cudart, nvrtc; print('✅ cuda-python OK')"
!pip install -q tensorrt==8.6.1 --extra-index-url https://pypi.nvidia.com
!python -c "import tensorrt as trt; print('✅ TRT', trt.__version__)"

In [ ]:
# ── Fix: upgrade typing_extensions so FastAPI can import 'Sentinel' ──────────
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-500:])
    return r.returncode

print('Step 1/3 — Upgrading typing_extensions to >=4.6.0 ...')
rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'typing_extensions>=4.6.0', '--upgrade'])
print('✅ done' if rc == 0 else '❌ failed')

print('Step 2/3 — Reinstalling FastAPI with compatible version ...')
rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'fastapi>=0.100.0', '--upgrade', '--force-reinstall', '--no-deps'])
print('✅ done' if rc == 0 else '❌ failed')

rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'starlette>=0.27.0', '--upgrade'])

print('Step 3/3 — Verifying fix ...')
try:
    import importlib, sys as _sys
    for mod in list(_sys.modules.keys()):
        if 'fastapi' in mod or 'typing_extensions' in mod or 'starlette' in mod:
            del _sys.modules[mod]
    import fastapi
    print(f'✅ FastAPI {fastapi.__version__} imports successfully!')
except Exception as e:
    print(f'❌ Still failing: {e}')
    print('   → Try restarting the kernel and re-running this cell.')


---
## ✅ Cell 2 — Verify Environment

In [ ]:
import torch, shutil, sys

print("=" * 58)
print("  Environment Check")
print("=" * 58)
print(f"  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA ok : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

for pkg in ['fastapi','uvicorn','onnxruntime']:
    try:
        m = __import__(pkg)
        print(f"  {pkg:<14}: {m.__version__}")
    except ImportError as e:
        print(f"  {pkg:<14}: ❌ {e}")

# JPEG encoder check
jpeg_backend = "PIL (slowest)"
try:
    from turbojpeg import TurboJPEG; jpeg_backend = "TurboJPEG (fastest)"
except ImportError:
    try:
        import cv2; jpeg_backend = "OpenCV cv2 (fast)"
    except ImportError:
        pass
print(f"  JPEG enc        : {jpeg_backend}")
print(f"  FFmpeg  : {'✅' if shutil.which('ffmpeg') else '❌ NOT FOUND'}")
print("=" * 58)

---
## ⚙️ Cell 3 — Download Checkpoints

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
import os

PROJECT_ROOT = os.path.abspath(os.getcwd())
CKPT_DIR     = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Bridge checkpoint (.pt) ───────────────────────────────────────────────────
print("📥 Downloading bridge_best.pt...")
snapshot_download(
    repo_id="Darknsu/bridge_best_onnx",
    repo_type="dataset",
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
)
print("✅ bridge_best.pt downloaded.")

# ── Bridge checkpoint (.onnx) ─────────────────────────────────────────────────
print("\n📥 Downloading bridge_best.onnx...")
snapshot_download(
    repo_id="Darknsu/librispeech-full-dataset-model",
    repo_type="dataset",
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
    allow_patterns=["bridge_best.pt"],
)
print("✅ bridge_best.onnx downloaded.")

# ── Ditto TRT models ──────────────────────────────────────────────────────────
print("\n📥 Downloading Ditto TRT models (may take a few minutes)...")
snapshot_download(
    repo_id="digital-avatar/ditto-talkinghead",
    repo_type="model",
    local_dir=os.path.join(PROJECT_ROOT, "ditto-inference", "checkpoints"),
    local_dir_use_symlinks=False,
)
print("✅ Ditto models downloaded.")

---
## 🔧 Cell 4 — Configure Server

**Edit paths here.**  The `.pt` streaming bridge (with KV-cache) is the
default — it produces the best motion quality with adaptive batching.
Set `BRIDGE_CHUNK = 1` and use `.onnx` for maximum throughput (no KV-cache).

In [ ]:
import os, torch

PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

# ════════════════════════════════════════════════════
# 🔧 Server settings
# ════════════════════════════════════════════════════
SERVER_HOST = "0.0.0.0"
SERVER_PORT = 7860

# ════════════════════════════════════════════════════
# 🔑 Moshi
# ════════════════════════════════════════════════════
MOSHI_HF_REPO   = "kyutai/moshiko-pytorch-bf16"
MOSHI_WEIGHT    = None   # local path to skip HF download
MIMI_WEIGHT     = None
MOSHI_TOKENIZER = None

# ════════════════════════════════════════════════════
# 🔑 Bridge
# ════════════════════════════════════════════════════
BRIDGE_CKPT   = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.pt")   # .pt streaming
BRIDGE_CONFIG = os.path.join(PROJECT_ROOT, "bridge_module", "config.yaml")

# BRIDGE_CHUNK: how many Mimi tokens to batch before running the bridge.
# The adaptive flush timer (BRIDGE_FLUSH_TIMEOUT_MS) ensures partial batches
# are sent promptly even during pauses — so higher chunk sizes no longer
# cause long stalls.
# Recommended: 4 (one Moshi LM window = 320ms of audio).
BRIDGE_CHUNK = 4

# Maximum time (ms) to wait before flushing a partial token batch to Ditto.
# 100ms = flush at most 100ms after the first token of a new batch arrives.
BRIDGE_FLUSH_TIMEOUT_MS = 100

# ════════════════════════════════════════════════════
# 🔑 Ditto
# ════════════════════════════════════════════════════
DITTO_DATA_ROOT = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_trt_Ampere_Plus")
DITTO_CFG_PKL   = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_cfg",
                                "v0.4_hubert_cfg_trt.pkl")
DITTO_EMO             = 4    # emotion index 0-7
DITTO_SAMPLING_STEPS  = 10   # ↓ reduces diffusion latency (try 10–20 for speed)
DITTO_JPEG_QUALITY    = 75   # ↓ smaller JPEG = faster transfer

# ════════════════════════════════════════════════════
# ✅ Validate paths
# ════════════════════════════════════════════════════
print("\n📋 Path Check")
print("─" * 60)
errors = []
checks = [
    ("Bridge .pt",       BRIDGE_CKPT),
    ("Bridge config",    BRIDGE_CONFIG),
    ("Ditto data root",  DITTO_DATA_ROOT),
    ("Ditto cfg pkl",    DITTO_CFG_PKL),
]
for label, path in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'}  {label:<22} {path}")
    if not exists:
        errors.append(label)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n  Device          : {DEVICE}")
print(f"  Bridge chunk    : {BRIDGE_CHUNK} tokens ({BRIDGE_CHUNK*80}ms audio)")
print(f"  Bridge flush    : {BRIDGE_FLUSH_TIMEOUT_MS}ms adaptive timeout")
print(f"  Ditto steps     : {DITTO_SAMPLING_STEPS} (lower=faster)")

if errors:
    print(f"\n⚠️  {len(errors)} issue(s): {errors}")
else:
    print("\n✅ All checks passed — ready to launch!")

---
## 🔬 Cell 4b — Pipeline Import Health Check

Verifies all modules import correctly **before** launching the server.
Run this after any code changes to catch import errors early.

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.getcwd())
for p in [PROJECT_ROOT,
          os.path.join(PROJECT_ROOT, 'bridge_module'),
          os.path.join(PROJECT_ROOT, 'moshi-inference')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("🔬 Pipeline Import Health Check")
print("═" * 50)

checks = {
    "pipeline.sync_types":        "TaggedToken, TaggedFeatures, TaggedFrame, seq_pack",
    "pipeline.streaming_moshi":   "StreamingMoshiState",
    "pipeline.ditto_stream_adapter": "DittoStreamAdapter",
    "inference":                  "StreamingBridgeInference",
}

all_ok = True
for mod, symbols in checks.items():
    try:
        m = __import__(mod, fromlist=symbols.split(', '))
        for sym in symbols.split(', '):
            getattr(m, sym.strip())
        print(f"  ✅  {mod:<40}  [{symbols}]")
    except Exception as e:
        print(f"  ❌  {mod:<40}  ERROR: {e}")
        all_ok = False

# Check struct.pack for seq encoding
try:
    import struct
    b = struct.pack('>I', 12345)
    assert len(b) == 4
    print(f"  ✅  seq_pack (struct.pack >I)              [4-byte big-endian uint32]")
except Exception as e:
    print(f"  ❌  seq_pack check failed: {e}")
    all_ok = False

print("═" * 50)
print("\n✅ All imports OK — safe to launch server." if all_ok else
      "\n❌ Fix import errors above before launching.")

---
## 🚀 Cell 5a — Kill Any Existing Server

In [ ]:
import os, signal, time

def get_pids_on_port(port):
    hex_port = format(port, '04X')
    inodes = set()
    for path in ["/proc/net/tcp", "/proc/net/tcp6"]:
        try:
            with open(path) as f:
                for line in f.readlines()[1:]:
                    parts = line.split()
                    if len(parts) > 9 and parts[1].split(":")[1] == hex_port:
                        inodes.add(parts[9])
        except FileNotFoundError:
            pass
    pids = []
    for pid in os.listdir("/proc"):
        if not pid.isdigit(): continue
        try:
            for fd in os.listdir(f"/proc/{pid}/fd"):
                try:
                    link = os.readlink(f"/proc/{pid}/fd/{fd}")
                    if "socket:[" in link and link.split("[")[1].rstrip("]") in inodes:
                        pids.append(int(pid)); break
                except OSError: pass
        except OSError: pass
    return pids

print("🔪 Killing any existing server processes...")
killed = []
for pid in os.listdir("/proc"):
    if not pid.isdigit(): continue
    try:
        with open(f"/proc/{pid}/cmdline") as f:
            if "streaming_server.py" in f.read():
                os.kill(int(pid), signal.SIGKILL); killed.append(pid)
    except (OSError, PermissionError): pass

if killed: print(f"   💀 Killed PIDs: {', '.join(killed)}")
time.sleep(1)

remaining = get_pids_on_port(SERVER_PORT)
for pid in remaining:
    try: os.kill(pid, signal.SIGKILL); print(f"   💀 Killed port-holder PID {pid}")
    except: pass
if remaining: time.sleep(1)

if get_pids_on_port(SERVER_PORT):
    print(f"⚠️  Port {SERVER_PORT} still busy!")
else:
    print(f"✅ Port {SERVER_PORT} is free — safe to launch.")

---
## 🚀 Cell 5b — Launch Streaming Server

All three models load at startup (~60–120s). Once **All models loaded** appears, open the URL.

**v2 startup log additions to look for:**
- `CUDA stream created for Bridge isolation` — confirms dedicated CUDA stream
- `Adaptive bridge flush: 100ms` — confirms timeout-based batching
- `JPEG encoder: TurboJPEG` / `OpenCV cv2` / `PIL` — shows encoder selected

In [ ]:
import os, sys, subprocess, threading, time
from IPython.display import display, HTML

PROJECT_ROOT = os.path.abspath(os.getcwd())

# ── Environment overrides ──────────────────────────────────────────────────────
env_overrides = {
    "MOSHI_HF_REPO":           MOSHI_HF_REPO,
    "BRIDGE_CKPT":             BRIDGE_CKPT,
    "BRIDGE_CONFIG":           BRIDGE_CONFIG,
    "BRIDGE_CHUNK":            str(BRIDGE_CHUNK),
    "BRIDGE_FLUSH_TIMEOUT_MS": str(BRIDGE_FLUSH_TIMEOUT_MS),
    "DITTO_DATA_ROOT":         DITTO_DATA_ROOT,
    "DITTO_CFG_PKL":           DITTO_CFG_PKL,
    "DITTO_EMO":               str(DITTO_EMO),
    "DITTO_SAMPLING_STEPS":    str(DITTO_SAMPLING_STEPS),
    "DITTO_JPEG_QUALITY":      str(DITTO_JPEG_QUALITY),
}
if MOSHI_WEIGHT:    env_overrides["MOSHI_WEIGHT"]    = MOSHI_WEIGHT
if MIMI_WEIGHT:     env_overrides["MIMI_WEIGHT"]     = MIMI_WEIGHT
if MOSHI_TOKENIZER: env_overrides["MOSHI_TOKENIZER"] = MOSHI_TOKENIZER

env = {**os.environ, **env_overrides}

# ── Public URL ────────────────────────────────────────────────────────────────
hostname   = os.environ.get("RUNPOD_POD_HOSTNAME",
             os.environ.get("RUNPOD_POD_ID", "localhost"))
PUBLIC_URL = (f"https://{hostname}-{SERVER_PORT}.proxy.runpod.net"
              if hostname != "localhost"
              else f"http://localhost:{SERVER_PORT}")

# ── Launch server ─────────────────────────────────────────────────────────────
server_script = os.path.join(PROJECT_ROOT, "streaming_server.py")
cmd = [sys.executable, "-u", server_script,
       "--host", SERVER_HOST, "--port", str(SERVER_PORT),
       "--bridge-flush-ms", str(BRIDGE_FLUSH_TIMEOUT_MS)]

print(f"🚀 Launching server  →  {PUBLIC_URL}\n")

proc = subprocess.Popen(
    cmd, cwd=PROJECT_ROOT, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

_ready = threading.Event()

def _stream_logs():
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
        if "All models loaded" in line or "Application startup complete" in line:
            _ready.set()
    print("\n⚠️  Server process exited.", flush=True)

threading.Thread(target=_stream_logs, daemon=True).start()

print("⏳ Waiting for models to load (1–2 min)…\n")
if _ready.wait(timeout=360):
    print("\n" + "═" * 62)
    print(f"  ✅  Server ready!")
    print(f"  🌐  {PUBLIC_URL}")
    print("═" * 62)
    display(HTML(
        f'<a href="{PUBLIC_URL}" target="_blank" '
        f'style="font-size:1.4em;font-weight:bold;color:#4f8ef7;">'
        f'▶ Open LiveAvatar ({PUBLIC_URL})</a>'
    ))
else:
    print("❌ Timed out — check logs above.")

# Keep cell alive so logs keep streaming
print("\n📡 Live logs (⬛ Interrupt to stop):\n")
try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate(); proc.wait()
    print("\n🛑 Server stopped.")

---
## 🛑 Cell 6 — Stop Server

In [ ]:
try:
    proc.terminate(); proc.wait(timeout=10)
    print("✅ Server stopped.")
except NameError:
    print("Server was not running.")
except Exception as e:
    print(f"Error: {e}"); proc.kill()

---
## 📊 Cell 7 — GPU Memory Check

In [ ]:
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated(0) / 1e9
    reserv = torch.cuda.memory_reserved(0)  / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🔬 GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Allocated : {alloc:.2f} GB")
    print(f"  Reserved  : {reserv:.2f} GB")
    print(f"  Total     : {total:.2f} GB")
    print(f"  Free      : {total - reserv:.2f} GB")
else:
    print("CUDA not available.")

---
## 🔬 Cell 8 — Live Server Health Check

Poll `/health` while the server is running to monitor pipeline status.

In [ ]:
import urllib.request, json, time

HEALTH_URL = f"http://localhost:{SERVER_PORT}/health"

print(f"🩺 Checking: {HEALTH_URL}")
try:
    with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
        data = json.loads(r.read())
    print("\nServer Health:")
    print(f"  Status         : {data.get('status', '?')}")
    print(f"  Device         : {data.get('device', '?')}")
    print(f"  Moshi loaded   : {'✅' if data.get('moshi_loaded') else '❌'}")
    print(f"  Moshi busy     : {'🔴 YES (session active)' if data.get('moshi_busy') else '✅ free'}")
    print(f"  Bridge loaded  : {'✅' if data.get('bridge_loaded') else '❌'}")
    print(f"  Bridge chunk   : {data.get('bridge_chunk', '?')} tokens")
    print(f"  Bridge flush   : {data.get('bridge_flush_ms', '?')}ms")
    print(f"  Ditto per-ses  : {'✅' if data.get('ditto_per_session') else '?'}")
except Exception as e:
    print(f"❌ Could not reach server: {e}")
    print("   Make sure Cell 5b is running.")

---
<div style="background:#080c14;padding:20px;border-radius:10px;text-align:center;border:1px solid rgba(79,142,247,.2);">
  <h3 style="color:#4f8ef7;margin:0;">⚡ Quick Reference — v2</h3>
  <table style="color:#64748b;margin:14px auto;border-collapse:collapse;font-family:sans-serif;font-size:13px;">
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Install</td>
        <td style="padding:5px 20px;">→ Cell 1 (once per session)</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Download</td>
        <td style="padding:5px 20px;">→ Cell 3 (.pt + Ditto TRT)</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Configure</td>
        <td style="padding:5px 20px;">→ Cell 4 (set paths &amp; options)</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Health check</td>
        <td style="padding:5px 20px;">→ Cell 4b 🔬 (import verification)</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Kill old server</td>
        <td style="padding:5px 20px;">→ Cell 5a</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Launch server</td>
        <td style="padding:5px 20px;">→ Cell 5b 🚀</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Stop server</td>
        <td style="padding:5px 20px;">→ Cell 6 🛑</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">GPU memory</td>
        <td style="padding:5px 20px;">→ Cell 7 📊</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Live health</td>
        <td style="padding:5px 20px;">→ Cell 8 🩺</td></tr>
  </table>
  <p style="color:#4f8ef7;font-size:12px;margin-top:10px;">
    Browser: Upload portrait → Connect → Start Speaking → Live talking head + audio + lip sync
  </p>
</div>